In [2]:
import numpy as np
import pandas as pd
import psycopg
from sqlalchemy import create_engine
import dotenv
import os
import sqlite3

In [3]:
nba = pd.read_csv('ASA All NBA Raw Data.csv', low_memory=False)

In [4]:
pd.set_option('display.max_rows', 81)
nba.head(3).T

,0,1,2
game_id,202202170BRK,202202170BRK,202202170BRK
game_date,2022-02-17,2022-02-17,2022-02-17
OT,0,0,0
H_A,A,A,A
Team_Abbrev,WAS,WAS,WAS
Team_Score,117,117,117
Team_pace,94.5,94.5,94.5
Team_efg_pct,0.627,0.627,0.627
Team_tov_pct,13.5,13.5,13.5
Team_orb_pct,22.9,22.9,22.9


## Database Normalization
### First normal form:

1. **All tables must have a primary key**: In this table, `game_id` and `player_id` together are unique on every row, and so they form primary key.

2. **All the data must be atomic**: Inactives is non-atomic.

3. **No repeating groups problem**: We can't solve the non-atomicity problem by creating separate columns if this leads to arbitrary ordering language in the column names (for example, `Inactive1`, `Inactive2`, etc.) and if it leads to a lot of missing data (there would be an `Inactive7` which would be missing any time a team has less than 7 inactive players).

In [5]:
nba = nba.drop(['Inactives'], axis=1)
nba.head(3).T

,0,1,2
game_id,202202170BRK,202202170BRK,202202170BRK
game_date,2022-02-17,2022-02-17,2022-02-17
OT,0,0,0
H_A,A,A,A
Team_Abbrev,WAS,WAS,WAS
Team_Score,117,117,117
Team_pace,94.5,94.5,94.5
Team_efg_pct,0.627,0.627,0.627
Team_tov_pct,13.5,13.5,13.5
Team_orb_pct,22.9,22.9,22.9


### Functional Dependence
Let X and Y be columns in a data table. Y is functionally dependent on X if each value of X has exactly one value of Y.

That's pretty abstract. So here are some guidelines that help me:

1. This use of "function" is the exact same as the concept of a function from algebra and pre-calculus. A correspondence f(x)=y is a function if each value of x has only one associated value of y.

2. X is either a primary key, or something that should be a primary key in another table.

For example, `game_date` (Y) is functionally dependent on `game_id` (X) because one `game_id` takes place on exactly one date.

### Second normal form:
In this table the primary key is a superkey consisting of two columns: `game_id` and `player_id`. 

2NF is violated if any columns are functionally dependent on part of the primary key but not the entire primary key. This can only happen if the primary key is a superkey.

In [6]:
games = nba[['game_id', 'game_date', 'OT', 'season']].drop_duplicates()
games.head(10)

,game_id,game_date,OT,season
0,202202170BRK,2022-02-17,0,2022
26,202202170CHO,2022-02-17,2,2022
48,202202170LAC,2022-02-17,0,2022
71,202202170MIL,2022-02-17,0,2022
95,202202170NOP,2022-02-17,0,2022
120,202202160BOS,2022-02-16,0,2022
121,202202140WAS,2022-02-14,0,2022
122,202202040DET,2022-02-04,0,2022
123,202202110DET,2022-02-11,0,2022
124,202202100DET,2022-02-10,0,2022


In [7]:
players = nba[['player_id', 'player']].drop_duplicates()
players.head(10)

,player_id,player
0,kispeco01,Corey Kispert
1,kuzmaky01,Kyle Kuzma
2,caldwke01,Kentavious Caldwell-Pope
3,netora01,Raul Neto
4,bryanth01,Thomas Bryant
5,avdijde01,Deni Avdija
6,smithis01,Ish Smith
7,hachiru01,Rui Hachimura
8,gillan01,Anthony Gill
9,gaffoda01,Daniel Gafford


### Third normal form:
3NF is violated if there are "transitive dependencies", that is, functional dependence between columns when neither column is part of the primary key.

In [8]:
nba.columns

Index(['game_id', 'game_date', 'OT', 'H_A', 'Team_Abbrev', 'Team_Score',
       'Team_pace', 'Team_efg_pct', 'Team_tov_pct', 'Team_orb_pct',
       'Team_ft_rate', 'Team_off_rtg', 'Opponent_Abbrev', 'Opponent_Score',
       'Opponent_pace', 'Opponent_efg_pct', 'Opponent_tov_pct',
       'Opponent_orb_pct', 'Opponent_ft_rate', 'Opponent_off_rtg', 'player',
       'player_id', 'starter', 'mp', 'fg', 'fga', 'fg_pct', 'fg3', 'fg3a',
       'fg3_pct', 'ft', 'fta', 'ft_pct', 'orb', 'drb', 'trb', 'ast', 'stl',
       'blk', 'tov', 'pf', 'pts', 'plus_minus', 'did_not_play', 'is_inactive',
       'ts_pct', 'efg_pct', 'fg3a_per_fga_pct', 'fta_per_fga_pct', 'orb_pct',
       'drb_pct', 'trb_pct', 'ast_pct', 'stl_pct', 'blk_pct', 'tov_pct',
       'usg_pct', 'off_rtg', 'def_rtg', 'bpm', 'season', 'minutes',
       'double_double', 'triple_double', 'DKP', 'FDP', 'SDP', 'DKP_per_minute',
       'FDP_per_minute', 'SDP_per_minute', 'pf_per_minute', 'ts',
       'last_60_minutes_per_game_starting', 'la

In [9]:
team_game = nba[['game_id', 'Team_Abbrev', 'H_A', 'Team_Score',
       'Team_pace', 'Team_efg_pct', 'Team_tov_pct', 'Team_orb_pct',
       'Team_ft_rate', 'Team_off_rtg', 'Opponent_Abbrev']].drop_duplicates()
team_game.head(10)

,game_id,Team_Abbrev,H_A,Team_Score,Team_pace,Team_efg_pct,Team_tov_pct,Team_orb_pct,Team_ft_rate,Team_off_rtg,Opponent_Abbrev
0,202202170BRK,WAS,A,117,94.5,0.627,13.5,22.9,0.157,123.8,BRK
13,202202170BRK,BRK,H,103,94.5,0.483,13.1,33.3,0.191,109.0,WAS
26,202202170CHO,MIA,A,111,88.8,0.471,11.1,26.8,0.147,103.4,CHO
37,202202170CHO,CHO,H,107,88.8,0.453,13.6,28.1,0.221,99.7,MIA
48,202202170LAC,HOU,A,111,103.7,0.533,15.3,24.0,0.154,107.1,LAC
59,202202170LAC,LAC,H,142,103.7,0.652,13.4,34.3,0.239,136.9,HOU
71,202202170MIL,PHI,A,123,99.9,0.568,10.7,22.0,0.261,123.1,MIL
84,202202170MIL,MIL,H,120,99.9,0.516,6.1,24.5,0.232,120.1,PHI
95,202202170NOP,DAL,A,125,94.2,0.652,7.9,18.4,0.220,132.6,NOP
107,202202170NOP,NOP,H,118,94.2,0.565,11.4,35.4,0.152,125.2,DAL


In [10]:
player_game = nba.drop(['game_date', 'OT', 'season', 'player', 'Team_Abbrev', 'H_A', 'Team_Score',
       'Team_pace', 'Team_efg_pct', 'Team_tov_pct', 'Team_orb_pct',
       'Team_ft_rate', 'Team_off_rtg', 'Opponent_Abbrev', 'Opponent_Score',
       'Opponent_pace', 'Opponent_efg_pct', 'Opponent_tov_pct',
       'Opponent_orb_pct', 'Opponent_ft_rate', 'Opponent_off_rtg'], axis=1)
player_game.head(10)

,game_id,player_id,starter,mp,fg,fga,fg_pct,fg3,fg3a,fg3_pct,...,pf_per_minute,ts,last_60_minutes_per_game_starting,last_60_minutes_per_game_bench,PG%,SG%,SF%,PF%,C%,active_position_minutes
0,202202170BRK,kispeco01,1,32:30,6,9,0.667,4,6,0.667,...,0.061538,9.00,31.716667,22.017778,1.0,36.0,60.0,4.0,0.0,46.253586
1,202202170BRK,kuzmaky01,1,30:16,2,7,0.286,0,3,0.000,...,0.099119,7.44,34.324000,18.475954,0.0,0.0,4.0,85.0,11.0,52.152590
2,202202170BRK,caldwke01,1,25:26,3,7,0.429,1,3,0.333,...,0.000000,7.00,29.820290,16.051693,0.0,32.0,67.0,0.0,0.0,47.021807
3,202202170BRK,netora01,1,20:40,5,7,0.714,1,1,1.000,...,0.048387,7.88,29.920833,14.603922,90.0,10.0,0.0,0.0,0.0,27.603314
4,202202170BRK,bryanth01,1,14:04,5,6,0.833,0,1,0.000,...,0.000000,6.88,20.095833,14.538095,0.0,0.0,0.0,0.0,100.0,36.472537
5,202202170BRK,avdijde01,0,30:50,4,10,0.400,1,3,0.333,...,0.097297,11.76,33.812500,23.658333,0.0,7.0,62.0,31.0,0.0,54.944529
6,202202170BRK,smithis01,0,27:20,7,10,0.700,1,3,0.333,...,0.109756,10.00,20.776017,11.183333,100.0,0.0,0.0,0.0,0.0,28.083850
7,202202170BRK,hachiru01,0,27:15,8,15,0.533,2,2,1.000,...,0.036697,15.88,29.610183,15.938596,0.0,0.0,33.0,57.0,10.0,53.836511
8,202202170BRK,gillan01,0,18:41,4,7,0.571,2,2,1.000,...,0.267618,8.32,12.768456,6.873016,0.0,0.0,16.0,70.0,14.0,52.255931
9,202202170BRK,gaffoda01,0,12:58,2,5,0.400,0,0,0.000,...,0.231362,5.88,20.789815,4.350000,0.0,0.0,0.0,0.0,100.0,36.472537


In [11]:
# postgres does not like capital letters in column names
player_game.columns = player_game.columns.str.lower()
team_game.columns = team_game.columns.str.lower()
players.columns = players.columns.str.lower()
games.columns = games.columns.str.lower()

,game_id,team_abbrev,h_a,team_score,team_pace,team_efg_pct,team_tov_pct,team_orb_pct,team_ft_rate,team_off_rtg,opponent_abbrev
0,202202170BRK,WAS,A,117,94.5,0.627,13.5,22.9,0.157,123.8,BRK
13,202202170BRK,BRK,H,103,94.5,0.483,13.1,33.3,0.191,109.0,WAS
26,202202170CHO,MIA,A,111,88.8,0.471,11.1,26.8,0.147,103.4,CHO
37,202202170CHO,CHO,H,107,88.8,0.453,13.6,28.1,0.221,99.7,MIA
48,202202170LAC,HOU,A,111,103.7,0.533,15.3,24.0,0.154,107.1,LAC
...,...,...,...,...,...,...,...,...,...,...,...
112068,202002270GSW,GSW,H,86,104.8,0.481,23.6,12.2,0.113,82.1,LAL
112079,202002290PHO,GSW,A,115,98.6,0.523,9.0,28.9,0.276,116.6,PHO
112090,202003010GSW,GSW,H,110,100.2,0.522,17.4,38.3,0.191,109.8,WAS
112101,202003030DEN,GSW,A,116,94.4,0.622,10.7,12.8,0.171,122.9,DEN
